## Libraries, Local Server, and Datasets

In [ ]:
from openai import OpenAI
import pandas as pd
from tqdm import tqdm
import time
import os
import json

# optional
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

In [ ]:
client = OpenAI(
    base_url="",
    api_key="lm-studio"  # dummy, required
)

In [ ]:
#observations_99pct = pd.read_csv(DATA_PROCESSED / "observations_99pct.csv")
#observations_above_99pct = pd.read_csv(DATA_PROCESSED / "observations_above_99pct.csv")


## Prompt Setup

### Identifier

In [ ]:
system_prompt = """You are a financial analyst. You output ONLY valid JSON. 
No explanations. No preamble. No markdown. No code blocks. 
The very first character of your response must be { and the very last must be }.

## TASK
Evaluate the ENTIRE earnings call Q&A text provided by the user and extract 
its forward-looking economic context.

## OUTPUT — STRICT RULES
- Output ONLY a single JSON object
- Do NOT write anything before { or after }
- Do NOT use markdown, backticks, or code fences
- Do NOT explain your reasoning
- Numeric values: floats, exactly 2 decimal places
- Categorical values: exact strings from the allowed lists only

## BAD OUTPUT (never do this):
Here is my analysis of the earnings call:
{"forward_looking_intensity": 0.60, ...}
Based on the above, the company seems optimistic.

## GOOD OUTPUT (always do this):
{"forward_looking_intensity": 0.60, ...}

---

## JSON SCHEMA

{
  "forward_looking_intensity": 0.00,
  "specificity": 0.00,
  "economic_substance": 0.00,
  "tone": 0.00,
  "certainty": 0.00,
  "context_summary": {
    "main_focus": "",
    "secondary_focus": "",
    "managerial_horizon": "",
    "overall_outlook": ""
  }
}

---

## STEP 1 — SCORE forward_looking_intensity (UNCONDITIONAL)

What share of the FULL Q&A is focused on future expectations, plans, or forecasts?

This is a ONE-SIDED measure. It asks only: how much of the discussion is forward-looking?
It does NOT ask how optimistic or specific the forward-looking content is — that is Step 2.
A low score means forward-looking content is limited. It says nothing about what the rest of the discussion is.

### How to score (follow this process exactly):

1. Estimate: roughly what percentage of the full Q&A is forward-looking content?
2. Convert that percentage directly to a decimal:
   10% → 0.10
   25% → 0.25
   40% → 0.40
   55% → 0.55
   70% → 0.70
   85% → 0.85

SCALE EXAMPLES — Rough guidelines for interpretation; ,DO NOT anchor your score to these examples, they are just for intuition:
- 0.08 = isolated future reference
- 0.28 = forward-looking minority
- 0.47 = roughly balanced
- 0.61 = forward-looking majority
- 0.83 = almost entirely forward-looking
- 1 = fully forward-looking.
---

## STEP 2 — SCORE conditional dimensions

All dimensions below are evaluated ONLY on the forward-looking portions identified in Step 1.
They measure the QUALITY of forward-looking content, not its quantity.

If forward_looking_intensity = 0.00, set all conditional dimensions to 0.00 and all
categoricals to "none".

### specificity
How concrete, detailed, and verifiable is the forward-looking content?

Penalize: vague language, absence of numbers or timeframes, generic statements.


### economic_substance
Do forward-looking statements contain value-relevant economic drivers investors can act on?

Penalize: generic strategy, mission statements, descriptive outlook without economic implications.
CRITICAL CHECK: Could an investor update a financial model or their investment decisions from this content? If not, economic_substance should be low.

### tone
Sentiment of forward-looking statements ONLY. Does not reflect past performance or current state. Where -1 is extremely negative and +1 is extremely positive. Score must be within the range [-1,1].


### certainty
How strong is managerial conviction in forward-looking statements?

Examples of high certainty cues: "will", "we expect", "we are confident", "we are on track"
Examples of low certainty cues: "might", "could", "hope", "uncertain", "subject to", "if conditions allow"

Penalize: frequent modal verbs, qualifications, uncertainty language.
NOTE: Low quantity of forward-looking content does NOT reduce certainty — score the quality of what is said.

---

## CATEGORICAL FIELDS

Assign based ONLY on forward-looking discussion.

main_focus — output EXACTLY ONE of the following strings and nothing else:
"strategy", "demand", "costs", "revenue", "supply", "investment",
"risk", "product", "market", "competition", "regulation", "operations"

Do NOT invent new labels. Do NOT use any value outside this list.
If the topic does not fit perfectly, pick the closest match from the list above.

BAD:  "main_focus": "renewables"      GOOD: "main_focus": "investment"
BAD:  "main_focus": "loan demand"     GOOD: "main_focus": "demand"
BAD:  "main_focus": "growth"          GOOD: "main_focus": "revenue"
BAD:  "main_focus": "macro_environment" GOOD: "main_focus": "risk"
BAD:  "main_focus": "technology"      GOOD: "main_focus": "strategy"

secondary_focus — output EXACTLY ONE of the following strings and nothing else:
"strategy", "demand", "costs", "revenue", "supply", "investment",
"risk", "product", "market", "competition", "regulation", "operations", "none"

Must differ from main_focus. Do NOT invent new labels. Do NOT combine multiple topics.
If no clear secondary topic exists, output "none".

BAD:  "secondary_focus": "capital spending, grid modernization"  GOOD: "secondary_focus": "investment"
BAD:  "secondary_focus": "M&A and capital allocation"           GOOD: "secondary_focus": "investment"
BAD:  "secondary_focus": "retirement services"                  GOOD: "secondary_focus": "revenue"

managerial_horizon — assign based on actual time references in the text:
- "short_term" = next 1–2 quarters explicitly discussed
- "medium_term" = 1–2 year horizon
- "long_term" = 3+ years, structural or multi-cycle outlook
- "mixed" = multiple distinct horizons discussed
- "none" = no temporal framing present


overall_outlook — output EXACTLY ONE of:
"negative", "neutral", "positive", "none"

---

## FINAL CHECK before outputting

1. Is my FLI score the result of an actual percentage estimate?
2. Is specificity 0.40 or below if there were no numbers or explicit timeframes?
3. Is economic_substance 0.50 or below if an investor could not update a model  from this?
4. Is managerial_horizon based on actual time language in the text, not defaulting to "medium_term"?
5. Are my scores spread across the range, or are they all clustered in a narrow band (e.g. 0.40–0.60) without justification?
6. Are main_focus and secondary_focus exact strings from the allowed lists, with no invented labels?"""



## LLM Functions

In [ ]:
def build_messages(dateutc, text, prompt):
    
    context_block = f"""Context:
Date of event: {dateutc}

Q&A Text:
{text}
"""

    return [
        {"role": "system", "content": prompt},
        {"role": "user", "content": context_block}
    ]

In [ ]:
def run_sample(text, dateutc, model, prompt, max_tokens, temperature=0):
    start = time.time()

    try:
        response = client.chat.completions.create(
            model=model,
            messages=build_messages(dateutc, text, prompt),
            temperature=temperature,
            max_tokens=max_tokens  # IMPORTANT: increase for testing
        )

        end = time.time()

        output = response.choices[0].message.content

        return {
            "output": output,
            "time_sec": end - start,
            "tokens": response.usage.total_tokens,
            "finish_reason": response.choices[0].finish_reason
        }

    except Exception as e:
        return {
            "output": None,
            "time_sec": None,
            "tokens": None,
            "finish_reason": None,
            "error": str(e)
        }

In [ ]:
def benchmark(df, df_name, model, prompt, n, max_tokens=2000, xid = None, temperature=0):
    os.makedirs("benchmarks/partial", exist_ok=True)
    os.makedirs("benchmarks/final", exist_ok=True)

    if n > len(df) or n is None:
        samples = df
    else:
        samples = df.sample(n, random_state=2000)

    results = []

    # --- Create run identifiers ONCE ---
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_model = model.replace("/", "_")

    filename = DATA_OUTPUT / f"final/{df_name}_{timestamp}.csv"
    partial_filename = DATA_OUTPUT / f"partial/{df_name}_{timestamp}_partial.csv"
    for i, row in enumerate(tqdm(samples.itertuples(index=False)), 1):
        text = row.transcript_text
        dateutc = getattr(row, "mostimportantdateutc", None)

        res = run_sample(
            text,
            dateutc=dateutc,
            model=model,
            prompt=prompt,
            max_tokens=max_tokens,
            temperature = temperature
        )

        # --- Safety guard ---
        if res is None:
            res = {}

        # Ensure required keys exist
        res.setdefault("finish_reason", None)
        res.setdefault("output", None)
        res.setdefault("time_sec", None)
        res.setdefault("tokens", None)

        # --- Add metadata ---
        res["transcript_id"] = getattr(row, "transcriptid", None)
        res['input_token_length'] = getattr(row, 'llama_tokens', None)
        res["date"] = getattr(row, "mostimportantdateutc", None)
        res['companyid'] = getattr(row, 'companyid', None)

        if xid is not None:
            res[xid] = getattr(row, xid, None)

        results.append(res)

        # --- Save every 10 iterations ---
        if i % 10 == 0:
            pd.DataFrame(results).to_csv(partial_filename, index=False)

    results_df = pd.DataFrame(results)

    # --- Diagnostics ---
    n_total = len(results_df)
    n_length = (results_df["finish_reason"] == "length").sum()
    pct_length = (results_df["finish_reason"] == "length").mean()

    time_quantiles = results_df.time_sec.quantile([0.25, 0.5, 0.75, 0.95, 0.99])

    n_missing = results_df["output"].isna().sum()
    pct_missing = results_df["output"].isna().mean()

    # --- Final save ---
    results_df.to_csv(filename, index=False)

    print("\n=== Benchmark Results ===")
    print(f"Saved to: {filename}")
    print(f"Total samples: {n_total}")
    print(f"Truncated (length): {n_length} ({pct_length:.2%})")
    print(f"Missing output: {n_missing} ({pct_missing:.2%})")
    print(f"Avg time: {results_df.time_sec.mean():.2f}s")
    print(f'Expected Hours needed for full dataset: {11009 * results_df.time_sec.mean() / 3600:.2f} hours')



    print("\n=== Time Quantiles ===")
    for q, val in time_quantiles.items():
        print(f"{int(q*100)}th percentile: {val:.2f}s")    
    print(f"Max time: {results_df.time_sec.max():.2f}s")
  
    print("\n=== Token Distributions ===")
    print(f"Avg tokens (input): {results_df.tokens.mean():.0f}")
    print(f"Min tokens (input): {results_df.tokens.min():.0f}")
    print(f"Max tokens (input): {results_df.tokens.max():.0f}")


    print("\n=== Output Validity ===")
    print("\nFinish reason distribution:")
    print(results_df["finish_reason"].value_counts(normalize=True))

    return results_df

## Batching

In [ ]:
data = pd.read_csv(DATA_PROCESSED / "observations_above_99pct.csv")

## MODELS

In [ ]:
df_name = "qwen_3"
obs = len(data)



print(f"Running model on {obs} samples...")
benchmark(data, df_name=df_name, model="qwen/qwen3-4b-2507",prompt= system_prompt, n = obs, max_tokens= 2500, temperature=0.0)

print("Finished")